# HealthBrowse: Llama 3 + QLoRA on free Colab

Run a small **General** extraction/citation-format experiment with Llama and QLoRA. It uses the six-deck JSON schema and follow-up format; it does not supply clinically reviewed reasoning targets. This notebook runs downloaded model weights; it makes **no paid inference API calls**. Hugging Face is used to download the base weights, subject to Meta's access terms.

**Status:** prepared workflow; no training results or medically validated adapter are supplied. Free Colab GPUs and session duration are not guaranteed. This is an interactive research notebook, not a public web server or a replacement for hosting HealthBrowse.

Before starting:
1. Choose **Runtime → Change runtime type → GPU**. A T4 may be assigned, but is not guaranteed.
2. Obtain authorized access to [Meta-Llama-3-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct).
3. Add a Hugging Face **read** token to Colab Secrets as `HF_TOKEN`; grant this notebook access. Do not put a token in a code cell. Paid Inference Providers permission is unnecessary for weight downloads.
4. **No upload is needed for the default research demo.** Section 4 creates `train.jsonl`, `validation.jsonl`, and a separate `test.jsonl` from the repository’s MedlinePlus health-topic summaries. Targets are intact excerpts marked **not clinically reviewed**. An optional `reviewed_upload` mode still accepts your own reviewed data. No visitor chats or patient records.
5. Keep several GB of Drive space available for adapters, optimizer checkpoints and reports. Downloaded base weights stay on the temporary VM, not Drive.

Run cells in order. First use the short 50-step experiment; it tests the pipeline and does not establish answer quality.

## 1. Get the project code
This cell downloads the pinned source commit containing automatic data preparation into its own versioned folder. It reuses only a clean, matching checkout and leaves an older `/content/medical_chatbot` folder and all Drive files alone. A failed download can be retried by running the cell again. No runtime deletion is needed to fix an older checkout.

The source commit is recorded in the experiment configuration. To resume, use that same pinned commit and run settings. If this repository is private, Git must already have authorized access; do not paste a GitHub token into this notebook or a clone URL. Access failures are separate from missing-file errors. An uploaded source ZIP is not automatically treated as the pinned Git checkout.

**Dependency repair:** this is a text-only PyTorch notebook. Setup removes optional `torchvision` and `torchaudio` from this Colab runtime before installing the pinned training dependencies, then checks the actual training imports in a fresh interpreter. This prevents an incompatible Colab TorchVision installation from failing with `operator torchvision::nms does not exist`. TensorFlow/Flax auto-detection is disabled for this PyTorch workflow. No dataset, Drive run, or repository source file is removed.

If upgrading an already-used notebook, choose **Runtime → Restart session** before running setup. This clears cached imports while retaining installed packages, the source checkout and Drive files. If setup reports that it repaired dependencies but found cached imports, restart the session once and run from the top. Successful setup prints **Text training imports: OK** before you proceed to the GPU/data cells. This import check is not a GPU training or memory-capacity test.


In [ ]:
from pathlib import Path
import subprocess, sys, json, os, tempfile, re
SOURCE_COMMIT = '520b78eb7b93179f3f66a1e4acc77e2ff01db7d7'
SOURCE_URL = 'https://github.com/McChandra/medical_chatbot.git'
CACHE_ROOT = Path('/content')

def ensure_checkout(cache_root, commit, repository_url):
    if not re.fullmatch(r'[0-9a-f]{40}', commit):
        raise ValueError('Use a full source commit SHA.')
    cache_root = Path(cache_root)
    cache_root.mkdir(parents=True, exist_ok=True)
    destination = cache_root / ('healthbrowse-qlora-' + commit[:12])
    if not destination.exists():
        # Build in a new temporary directory; a failed fetch cannot leave a stale
        # final checkout. Never modify /content/medical_chatbot or saved Drive runs.
        with tempfile.TemporaryDirectory(prefix='hb-download-', dir=cache_root) as temporary:
            staging = Path(temporary) / 'repo'
            staging.mkdir()
            subprocess.run(['git','init','--quiet',str(staging)],check=True)
            subprocess.run(['git','-C',str(staging),'fetch','--quiet','--depth','1',repository_url,commit],check=True)
            subprocess.run(['git','-C',str(staging),'checkout','--quiet','--detach',commit],check=True)
            staging.rename(destination)
    if not (destination/'.git').is_dir():
        raise RuntimeError('The versioned path contains other files. Keep them and choose another CACHE_ROOT.')
    actual = subprocess.check_output(['git','-C',str(destination),'rev-parse','HEAD'],text=True).strip()
    if actual != commit:
        raise RuntimeError('The cached checkout has changed. Preserve it and choose another CACHE_ROOT.')
    if subprocess.check_output(['git','-C',str(destination),'status','--porcelain'],text=True).strip():
        raise RuntimeError('The cached checkout has local edits. Preserve them and choose another CACHE_ROOT.')
    return destination

REPO = ensure_checkout(CACHE_ROOT, SOURCE_COMMIT, SOURCE_URL)
INFERENCE = REPO / 'healthbrowse' / 'inference'
required = ['prepare_research_data.py','train_adapter.py','requirements.txt']
missing = [name for name in required if not (INFERENCE/name).is_file()]
if missing:
    raise RuntimeError('Incomplete pinned source checkout: ' + ', '.join(missing))
SOURCE_VERSION = SOURCE_COMMIT
# A prior notebook attempt may have cached imports from the old checkout.
for name in ['train_adapter','model_settings','prepare_research_data']:
    module = sys.modules.get(name)
    if module and Path(getattr(module,'__file__','')).parent != INFERENCE:
        del sys.modules[name]
print('Source version:', SOURCE_VERSION)
print('Source folder:', REPO)
print('Data preparation script: ready')
def prepare_text_environment(requirements):
    # This notebook trains text-only Llama. Colab's preinstalled image/audio
    # extensions may target a different torch ABI than the pinned torch 2.8.
    imported_before = [name for name in ['torch','transformers','torchvision','torchaudio','peft','bitsandbytes']
                       if name in sys.modules]
    os.environ['USE_TF'] = '0'
    os.environ['USE_FLAX'] = '0'
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchvision','torchaudio'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(requirements)],check=True)
    # Use a fresh interpreter: imported modules in this kernel may still refer
    # to the packages that pip just replaced or removed.
    smoke = (
        "import torch; "
        "from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, "
        "Trainer, TrainingArguments, DataCollatorForSeq2Seq, set_seed; "
        "from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training; "
        "import bitsandbytes; "
        "print('Text training imports: OK; torch=' + torch.__version__)"
    )
    result = subprocess.run([sys.executable,'-c',smoke],stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT,text=True)
    print(result.stdout,flush=True)
    if result.returncode:
        raise RuntimeError('Dependency import check failed before training. Full error tail:\n' + result.stdout[-12000:])
    if imported_before:
        raise RuntimeError('Dependencies are repaired, but this session has cached imports. '
                           'Choose Runtime > Restart session (not Disconnect and delete runtime), '
                           'then run this notebook from the top. Drive files and the source checkout remain.')

prepare_text_environment(INFERENCE/'requirements.txt')


## 2. Check the assigned GPU
The script chooses FP16 for T4-class GPUs and native BF16 where supported. The 2,048-token, rank-8 profile is a starting configuration, not a guarantee that every session will fit. If no GPU is available, stop and try an available authorized session later. Do not use keep-alive scripts or multiple accounts to bypass limits.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU assigned. Select a GPU runtime; availability depends on Colab.')
sys.path.insert(0,str(INFERENCE))
from train_adapter import precision_flags
props = torch.cuda.get_device_properties(0)
FLAGS = precision_flags(torch.cuda.get_device_capability()[0],torch.cuda.is_bf16_supported())
DTYPE = torch.bfloat16 if FLAGS['bf16'] else torch.float16
print({'GPU':props.name,'VRAM_GiB':round(props.total_memory/2**30,1),'dtype':str(DTYPE)})

## 3. Access the weights and save the experiment configuration
Mount only your own research Drive. Checkpoints remain there when the VM ends. This notebook does not upload weights publicly. To start a different experiment, use a **new RUN_NAME** instead of overwriting an old run.

In [ ]:
from google.colab import userdata, drive
from huggingface_hub import HfApi
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
assert os.environ['HF_TOKEN'], 'Add the read token to Colab Secrets, not notebook source.'
drive.mount('/content/drive')
RUN_NAME = 'llama3-general-research-002'  # New name for this updated experiment.
DATA_MODE = 'research_demo'  # Or 'reviewed_upload' for your own clinically reviewed targets.
assert DATA_MODE in {'research_demo','reviewed_upload'}
RUN_DIR = Path('/content/drive/MyDrive/HealthBrowse') / RUN_NAME
RUN_DIR.mkdir(parents=True,exist_ok=True)
MODEL_ID = 'meta-llama/Meta-Llama-3-8B-Instruct'
MAX_LENGTH = 2048
MAX_STEPS = 50  # Pipeline experiment; use a new run with -1 for epoch-based training.
EPOCHS = 2
SAVE_STEPS = 10
LORA_RANK = 8
config_path = RUN_DIR / 'experiment.json'
previous = json.loads(config_path.read_text()) if config_path.exists() else None
BASE_REVISION = previous['base_revision'] if previous else HfApi().model_info(MODEL_ID).sha
config = {'source_version':SOURCE_VERSION,'model':MODEL_ID,'base_revision':BASE_REVISION,
          'data_mode':DATA_MODE,'max_length':MAX_LENGTH,'max_steps':MAX_STEPS,'epochs':EPOCHS,
          'save_steps':SAVE_STEPS,'lora_rank':LORA_RANK}
if previous and previous != config:
    raise ValueError('Run settings/source differ. Restore the recorded source/settings or choose a new RUN_NAME.')
config_path.write_text(json.dumps(config,indent=2))
print(config)  # No credentials in this configuration.

## 4. Prepare the data automatically
With `DATA_MODE = 'research_demo'`, run the next cell. **Do not upload anything.** It creates `train.jsonl`, `validation.jsonl`, `test.jsonl`, and `dataset-manifest.json` inside your run's `data` folder in Drive.

This is a small **extractive formatting baseline** from public-domain MedlinePlus health-topic summaries. It copies complete short passages with source URLs and exact quotes. It does not generate new medical facts or claim clinical review. Six deck slots are present; areas without supplied evidence are empty. Labels come from the existing heuristic parser, so this is not a complete clinical training corpus. Only these topic summaries are used, not copyrighted encyclopedia/drug pages or linked documents. [Source and reuse policy](https://medlineplus.gov/about/using/usingcontent/).

Groups sharing canonical topics, aliases, publisher pages or identical selected passages stay in one split. Related diseases and near-duplicates still need review. The test split is saved but never used for training, validation loss, or the notebook comparison. Actual token lengths are checked before files are written; an overlength example is excluded intact rather than truncated.

For **your own clinically reviewed data**, set `DATA_MODE = 'reviewed_upload'` above with a different run name. Only that mode asks for the two reviewed files. Never change review flags merely to pass a check.

In [ ]:
from google.colab import files
from transformers import AutoTokenizer
from train_adapter import load_examples, validate_split
DATA_DIR = RUN_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)
TRAIN = DATA_DIR/'train.jsonl'
VALIDATION = DATA_DIR/'validation.jsonl'
RESEARCH_ONLY = DATA_MODE == 'research_demo'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID,revision=BASE_REVISION,trust_remote_code=False)
if RESEARCH_ONLY:
    from prepare_research_data import prepare
    dataset_manifest = prepare(
        REPO/'healthbrowse'/'dist'/'medline-topics.json', DATA_DIR,
        token_count=lambda row: len(tokenizer.apply_chat_template(row['messages'],tokenize=True)),
        max_tokens=MAX_LENGTH, tokenizer_id=MODEL_ID+'@'+BASE_REVISION)
    print('Research-only dataset prepared:',dataset_manifest['counts'])
    print('Test data stays untouched. Clinical review: not performed.')
else:
    if not (TRAIN.exists() and VALIDATION.exists()):
        uploaded = files.upload()
        if set(uploaded) != {'train.jsonl','validation.jsonl'}:
            raise ValueError('Select exactly train.jsonl and validation.jsonl.')
        for name, data in uploaded.items():
            target=DATA_DIR/name
            if target.exists() and target.read_bytes()!=data:
                raise ValueError('Existing data differs. Choose a new run instead of replacing it.')
        for name, data in uploaded.items():
            (DATA_DIR/name).write_bytes(data)
        del uploaded
train_rows = load_examples(TRAIN,research_only=RESEARCH_ONLY)
validation_rows = load_examples(VALIDATION,research_only=RESEARCH_ONLY)
validate_split(train_rows,validation_rows)
print({'training_examples':len(train_rows),'validation_examples':len(validation_rows),'split_check':'passed'})

## 5. Check token lengths and recover any interrupted checkpoint
Overlength examples fail explicitly. Shorten evidence thoughtfully, keep complete supporting passages, or select a new experiment with an appropriate length. Do not silently truncate clinical qualifiers or target citations.

Only resume your own checkpoints. A new run, changed data, or changed settings require a new output directory. If interrupted before the first complete checkpoint, keep the old run and begin a new one.

In [ ]:
from transformers.trainer_utils import get_last_checkpoint
from model_settings import validate_base
validate_base(MODEL_ID,BASE_REVISION,MAX_LENGTH)
lengths = [len(tokenizer.apply_chat_template(row['messages'],tokenize=True))
           for row in train_rows+validation_rows]
print({'longest_example_tokens':max(lengths),'limit':MAX_LENGTH})
if max(lengths)>MAX_LENGTH:
    raise ValueError('Overlength examples: revise the prepared data explicitly in a new run.')
OUTPUT = RUN_DIR/'adapter'
checkpoint = get_last_checkpoint(str(OUTPUT)) if OUTPUT.exists() else None
cmd = [sys.executable,str(INFERENCE/'train_adapter.py'),'--train',str(TRAIN),'--validation',str(VALIDATION),
       '--model',MODEL_ID,'--revision',BASE_REVISION,'--method','qlora','--output',str(OUTPUT),
       '--max-length',str(MAX_LENGTH),'--max-steps',str(MAX_STEPS),'--epochs',str(EPOCHS),
       '--save-steps',str(SAVE_STEPS),'--lora-rank',str(LORA_RANK)]
if RESEARCH_ONLY:
    cmd += ['--research-only']
if checkpoint:
    cmd += ['--resume-from-checkpoint',checkpoint]
    print('Resuming:',Path(checkpoint).name)
subprocess.run(cmd+['--validate-only'],check=True)

## 6. Train the adapter
Run the next cell only after Section 4 says **Research-only dataset prepared** (or your reviewed upload passes) and Section 5's checks pass. The default is a **50-step pipeline experiment**. Keep the notebook open and monitor it. It does not establish improved medical accuracy or clinical safety.

Checkpoints are written every 10 optimizer steps. After interruption, use the same source commit, Drive run name, data mode and settings to resume. If settings or data change, start a new run name. Free Colab resources and completion are not guaranteed.
Training output is streamed here and saved under `RUN_DIR/logs/`. A failure includes the last 80 output lines in the exception itself, so the underlying error is visible. Log headers record Python/package versions, without environment credentials. Existing logs are retained. A retry checks for the latest saved checkpoint; it never deletes partial output. If no checkpoint exists and the output folder is nonempty, preserve that failed run and inspect its log before choosing a new run name.


In [ ]:
from collections import deque
from datetime import datetime, timezone
import importlib.metadata
import uuid

def run_training(command, log_dir):
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True,exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    log_path = log_dir / ('training-' + stamp + '-' + uuid.uuid4().hex[:8] + '.log')
    packages = {}
    for name in ['torch','transformers','accelerate','peft','bitsandbytes','torchvision']:
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = 'not installed'
    tail = deque(maxlen=80)
    secrets = [os.environ[name] for name in ['HF_TOKEN','HUGGING_FACE_HUB_TOKEN'] if os.environ.get(name)]
    def redact(text):
        for value in secrets:
            text = text.replace(value,'[REDACTED]')
        return text
    print('Training log:',log_path)
    with log_path.open('w',encoding='utf-8') as log:
        log.write(json.dumps({'python':sys.version,'packages':packages})+'\n')
        log.flush()
        process = subprocess.Popen([command[0],'-u',*command[1:]],stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        try:
            for line in process.stdout:
                line = redact(line)
                print(line,end='',flush=True)
                log.write(line)
                log.flush()
                tail.append(line)
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
    if returncode:
        details = ''.join(tail)[-12000:] or '(The process produced no output.)'
        raise RuntimeError(f'Training exited with code {returncode}. Log: {log_path}\n'
                           'The actual training output is below. Share the traceback, not just the exit code.\n'
                           + details)
    return log_path

# Refresh resume state if this cell is retried after an interrupted attempt.
# Preserve all partial output; the trainer still checks data/settings before resume.
training_cmd = list(cmd)
if '--resume-from-checkpoint' in training_cmd:
    position = training_cmd.index('--resume-from-checkpoint')
    del training_cmd[position:position+2]
checkpoint = get_last_checkpoint(str(OUTPUT)) if OUTPUT.exists() else None
if checkpoint:
    training_cmd += ['--resume-from-checkpoint',checkpoint]
TRAINING_LOG = run_training(training_cmd,RUN_DIR/'logs')
assert (OUTPUT/'adapter_model.safetensors').is_file(), 'Training did not produce a final adapter; inspect the saved training log.'
print('Saved adapter:',OUTPUT)
print(json.loads((OUTPUT/'eval_results.json').read_text()))


## 7. Compare the base model and adapter on held-out questions
Use identical prompts and greedy generation. This samples at most five held-out validation rows. It is a sanity check, not a statistically adequate benchmark. Keep a separate untouched test split for the final research study.

The check below verifies parseability, deck IDs, source IDs and exact quote membership. A real quote may still fail to support a generated claim; manually assess entailment, completeness, readability and dangerous omissions. No score here certifies clinical safety.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from model_settings import validate_adapter
validate_adapter(OUTPUT,MODEL_ID,BASE_REVISION)
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                        bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=DTYPE)
base=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=BASE_REVISION,
    quantization_config=quant,torch_dtype=DTYPE,device_map={'':0},trust_remote_code=False)
model=PeftModel.from_pretrained(base,str(OUTPUT))
model.eval()
tokenizer.pad_token=tokenizer.eos_token

def generate(messages):
    inputs=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,
                                        return_tensors='pt',return_dict=True).to(model.device)
    count=inputs['input_ids'].shape[-1]
    budget=min(1024,model.config.max_position_embeddings-count)
    if budget<=0:
        raise ValueError('Prompt exceeds model context.')
    with torch.inference_mode():
        output=model.generate(**inputs,max_new_tokens=budget,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    tokens=output[0,count:]
    return tokenizer.decode(tokens,skip_special_tokens=True),len(tokens)>=budget

def structure_check(text,payload,truncated):
    if truncated:
        return {'passed':False,'reason':'output token limit reached'}
    try:
        data=json.loads(text)
        expected=[1,2,3,4,5,6] if payload['mode']=='search' else []
        if sorted(d['id'] for d in data['decks'])!=expected:
            raise ValueError('deck IDs')
        evidence={s['id']:s['text'] for s in payload['evidence']}
        claims=[c for p in [data['answer'],*data['decks']] for c in p['claims']]
        for claim in claims:
            if not claim['text'].strip() or not claim['support']:
                raise ValueError('missing text/support')
            for support in claim['support']:
                quote=support['quote']
                if len(quote.strip())<20 or quote not in evidence[support['id']]:
                    raise ValueError('unsupported quote')
        return {'passed':True,'claims':len(claims),'entailment_review':'not performed'}
    except (ValueError,KeyError,TypeError) as e:
        return {'passed':False,'reason':str(e)}

comparisons=[]
for row in validation_rows[:5]:
    messages=row['messages'][:-1]
    payload=json.loads(messages[1]['content'])
    with model.disable_adapter():
        before,before_cut=generate(messages)
    after,after_cut=generate(messages)
    item={'topic':row['topic'],'question':payload.get('question'),
          'base':before,'adapter':after,
          'base_structure':structure_check(before,payload,before_cut),
          'adapter_structure':structure_check(after,payload,after_cut)}
    comparisons.append(item)
    print(json.dumps(item,indent=2))
(RUN_DIR/'comparison.json').write_text(json.dumps(comparisons,indent=2))
print('Saved held-out comparison; clinical review is still required.')

## 8. Download the adapter and reports
This exports the adapter and tokenizer files, not the large base model or optimizer checkpoints. The complete checkpoints remain in Drive for recovery. Treat the exported adapter as research-only until evaluated; do not redistribute the underlying datasets without permission.

In [ ]:
import zipfile
archive=RUN_DIR/'Llama-3-HealthBrowse-General-QLoRA.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT.iterdir():
        if path.is_file() and path.suffix in {'.json','.safetensors','.model','.txt'}:
            z.write(path,'adapter/'+path.name)
    z.write(config_path,'experiment.json')
    for data_path in DATA_DIR.iterdir():
        if data_path.is_file() and data_path.suffix in {'.json','.jsonl'}:
            z.write(data_path,'data/'+data_path.name)
    if (RUN_DIR/'comparison.json').exists():
        z.write(RUN_DIR/'comparison.json','comparison.json')
files.download(str(archive))

## 9. Connect HealthBrowse after evaluation
This notebook does **not** activate AI on the public Site. Run the base and adapter on your own or authorized shared compute, behind an authenticated HTTPS endpoint. Keep HealthBrowse's retrieval, source checks and review pipeline; fine-tuning is not a substitute for those controls.

On that host, use the existing `inference/serve.py` with `MODEL_ID`, the pinned `MODEL_REVISION`, `ADAPTER_PATH` and a server-only `INFERENCE_API_KEY`. The current serving script loads full-precision weights, so its memory requirements are different from this notebook's 4-bit evaluation loader. Do not assume it fits the same free Colab GPU.

The Site then needs `MODEL_GENERAL_URL`, `MODEL_GENERAL_ID`, `MODEL_GENERAL_KEY` and `MODEL_GENERAL_CONTEXT_TOKENS=8192`, followed by a deployment and an authenticated end-to-end test. These are self-hosted endpoint credentials, not Featherless credentials.

Free Colab is not an always-on backend. Do not tunnel the public website through a free managed notebook or use keep-alive workarounds.

### Sources
- [QLoRA paper](https://arxiv.org/abs/2305.14314)
- [PEFT quantization guide](https://huggingface.co/docs/peft/v0.17.0/developer_guides/quantization)
- [Trainer checkpoints](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer)
- [Meta Llama 3 model and terms](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)
- [Colab FAQ and restrictions](https://research.google.com/colaboratory/faq.html)

Built with Meta Llama 3. No clinical certification, guaranteed factuality, or completed training result is claimed.